# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Load environment 
%load_ext dotenv
%dotenv 



In [14]:
import dask.dataframe as dd
dd

<module 'dask.dataframe' from '/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/__init__.py'>

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Load env variable 
price_data_dir = os.environ.get('PRICE_DATA')

# Use glob to find all files within the directory 
parquet_files = glob(os.path.join(price_data_dir, '**/*.parquet'),recursive=True)

# Get list of files to check
parquet_files

['../../05_src/data/prices/SFUN/SFUN_2015/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2015/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2012/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2012/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2013/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2013/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2014/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2014/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2020/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2020/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2018/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2018/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2011/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2011/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2016/part.0.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2016/part.1.parquet',
 '../../05_src/data/prices/SFUN/SFUN_2017/part.0.parquet

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [47]:
import dask.dataframe as dd

# Create a Dask DF from the preselected files
dd_feat = dd.read_parquet(parquet_files)


# Add lags for Close and Adj Close
dd_feat = dd_feat.map_partitions(
    lambda df: df.sort_values(['ticker', 'Date'])
    .groupby('ticker', group_keys=False)
    .apply(lambda x: x.assign(
        Close_lag_1=x['Close'].shift(1),
        Adj_Close_lag_1=x['Adj Close'].shift(1)
    ))
)

# Add returns
dd_feat['returns'] = (dd_feat['Close'] / dd_feat['Close_lag_1']) - 1

# Add hi_lo_range
dd_feat['hi_lo_range'] = dd_feat['High'] - dd_feat['Low']

# Persist the changes
dd_feat = dd_feat.persist()

# View first few rows 
print(dd_feat.head())


/var/folders/ml/sgrl0zy97gdf902pd3ct59480000gn/T/ipykernel_30921/372819962.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lambda df: df.sort_values(['ticker', 'Date'])


            Date       Open       High        Low      Close  Adj Close  \
38739 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
38740 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
38741 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
38742 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
38743 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

           Volume source ticker  Year  Close_lag_1  Adj_Close_lag_1   returns  \
38739  62546300.0  A.csv      A  1999          NaN              NaN       NaN   
38740  15234100.0  A.csv      A  1999    31.473534        27.068665 -0.082386   
38741   6577800.0  A.csv      A  1999    28.880543        24.838577  0.089783   
38742   5975600.0  A.csv      A  1999    31.473534        27.068665 -0.090909   
38743   4843200.0  A.csv      A  1999    28.612303        24.607880  0.026563   

       hi_lo_range  
38739     7.153078  
38740     2.280043  

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [48]:
import pandas as pd

# Convert Dask DataFrame to Pandas DataFrame
pd_feat = dd_feat.compute()

# Add 10-day moving average of returns
pd_feat['returns_ma_10'] = pd_feat.groupby('ticker')['returns'].rolling(window=10).mean().reset_index(level=0, drop=True)

# View first few rows of the Pandas DataFrame
print(pd_feat.head())

            Date       Open       High        Low      Close  Adj Close  \
38739 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
38740 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
38741 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
38742 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
38743 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

           Volume source ticker  Year  Close_lag_1  Adj_Close_lag_1   returns  \
38739  62546300.0  A.csv      A  1999          NaN              NaN       NaN   
38740  15234100.0  A.csv      A  1999    31.473534        27.068665 -0.082386   
38741   6577800.0  A.csv      A  1999    28.880543        24.838577  0.089783   
38742   5975600.0  A.csv      A  1999    31.473534        27.068665 -0.090909   
38743   4843200.0  A.csv      A  1999    28.612303        24.607880  0.026563   

       hi_lo_range  returns_ma_10  
38739     7.153078        

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?


Dask is designed to handle large data, allowing to perform operations in a distributed and parallel manner. For that reason, Dask can be more efficient and faster than Pandas, particularly for computationally intensive tasks like calculating moving averages. So it was not necessary to make the conversion. 

Using Pandas is simpler for small datasets, it is less scalable and computationally slower that Dask.  

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.